
# Rate My Recipe

## 1. Dataset Exploration

### Project Objective

The goal of this project is to investigate how recipes can be ranked fairly according to user ratings, taking into account both the average rating and the number of reviews.

### Dataset

The project uses the Food.com Recipes and Interactions dataset. It contains recipe information and user reviews and ratings.

The first step is to inspect the structure and quality of the available data before deciding how to prepare it for the ranking analysis.


In [2]:

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# Visualization settings
sns.set_theme(style="whitegrid")



## 2. Loading the Datasets

In this section, we load the two main CSV files into pandas DataFrames.

- `RAW_recipes.csv` contains information about the recipes.
- `RAW_interactions.csv` contains user interactions and ratings.

Loading the datasets allows us to inspect their structure, understand the available variables, and determine how the recipe information can be connected to user ratings.

The original CSV files are stored locally in the `data/` directory and are excluded from version control.


In [3]:

recipes = pd.read_csv("../data/RAW_recipes.csv")
interactions = pd.read_csv("../data/RAW_interactions.csv")

print("Recipes dataset loaded successfully.")
print("Interactions dataset loaded successfully.")


Recipes dataset loaded successfully.
Interactions dataset loaded successfully.



## 3. Initial Dataset Inspection

Before performing any data cleaning or analysis, we need to understand the structure of the two datasets.

In this section, we will:
- Check the number of rows and columns in each dataset.
- Inspect the column names to understand which variables are available.
- Display the first few rows to examine how the data is organised.

This initial inspection will help us identify the information needed to analyse recipe ratings and review counts.


In [4]:

print("Recipes dataset shape:", recipes.shape)
print("Interactions dataset shape:", interactions.shape)


Recipes dataset shape: (231637, 12)
Interactions dataset shape: (1132367, 5)


In [5]:

print("Recipe columns:")
print(recipes.columns.tolist())

print("\nInteraction columns:")
print(interactions.columns.tolist())


Recipe columns:
['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']

Interaction columns:
['user_id', 'recipe_id', 'date', 'rating', 'review']


In [6]:

display(recipes.head())


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'o...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourth...",autumn is my favorite time of year to cook! this recipe \r\ncan be prepared either spicy or swee...,"['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'o...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough into the bottom and sides of a 12 inch pizza pan'...",this recipe calls for the crust to be prebaked a bit before adding ingredients. feel free to cha...,"['prepared pizza crust', 'sausage patty', 'eggs', 'milk', 'salt and pepper', 'cheese']",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'main-dish', 'chili', 'crock-pot-slow-cooker', 'dietar...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add chopped onions to ground beef when almost brown and saut...",this modified version of 'mom's' chili was a hit at our 2004 christmas party. we made an extra l...,"['ground beef', 'yellow onions', 'diced tomatoes', 'tomato paste', 'tomato soup', 'rotel tomatoe...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'preparation', 'occasion', '...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,"['place potatoes in a large pot of lightly salted water and bring to a gentle boil', 'cook until...","this is a super easy, great tasting, make ahead side dish that looks like you spent a lot more t...","['spreadable cheese with garlic and herbs', 'new potatoes', 'shallots', 'parsley', 'tarragon', '...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion',...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,"['mix all ingredients& boil for 2 1 / 2 hours , or until thick', 'pour into jars', ""i use'old' g...",my dh's amish mother raised him on this recipe. he much prefers it over store-bought ketchup. it...,"['tomato juice', 'apple cider vinegar', 'sugar', 'salt', 'pepper', 'clove oil', 'cinnamon oil', ...",8


In [7]:

display(interactions.head())


,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for 15 minutes.Added a shake of cayenne and a pinch o...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall evening. Should have doubled it ;)<br/><br/>Secon..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not quite a whole package (10oz) of white chips. Great!
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunko. Everyone loved it.
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprinkling of black pepper. Yum!"


The dataset contains 231,637 recipes and 1,132,367 user interactions.

The recipes table contains recipe details, while the interactions table contains user ratings and reviews.

The tables can be linked using `recipes.id` and `interactions.recipe_id`. We will verify this relationship before proceeding with the analysis.


## 4. Review Distribution

In this section, we analyse how reviews are distributed across recipes.

We will calculate the number of reviews per recipe, examine the distribution of ratings, and count how many recipes have at least one review.

These results will help us understand the available data and decide whether a subset is necessary for the ranking analysis.


In [8]:

# Count the reviews received by each recipe
reviews_per_recipe = interactions.groupby("recipe_id").size()

print("Reviews per recipe:")
display(reviews_per_recipe.describe())

# Count the number of reviews for each rating
print("Rating distribution:")
display(interactions["rating"].value_counts().sort_index())

# Count recipes with at least one review
recipes_with_reviews = recipes["id"].isin(interactions["recipe_id"]).sum()

print("Recipes with at least one review:", recipes_with_reviews)
print("Recipes without reviews:", len(recipes) - recipes_with_reviews)


Reviews per recipe:


count    231637.000000
mean          4.888541
std          17.532481
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max        1613.000000
dtype: float64

Rating distribution:


rating
0     60847
1     12818
2     14123
3     40855
4    187360
5    816364
Name: count, dtype: int64

Recipes with at least one review: 231637
Recipes without reviews: 0


# RESULTS
### Number of Reviews per Recipe

- The distribution is highly skewed: the median is 2 reviews, 75% of recipes have 4 reviews or fewer, the mean is 4.89, and the maximum is 1,613.
- This suggests that average ratings alone may be misleading. A recipe with a single 5-star review could rank above a recipe with hundreds of reviews and an average rating of 4.9.

### Rating Distribution

- Ratings are concentrated at the top: 816,364 out of 1,132,367 interactions (approximately 72%) have a rating of 5.
- There are 60,847 ratings equal to 0, considerably more than ratings of 1 or 2. These zero ratings require further investigation to determine whether they represent missing ratings or valid values. Their treatment could affect the ranking results.


## 5. Review Count Thresholds

The number of reviews varies considerably across recipes. In this section, we count how many recipes have at least a given number of reviews.

This analysis will help us understand how the dataset is distributed and whether a minimum review threshold or a subset is appropriate for the ranking analysis.


In [9]:

for n in [1, 5, 10, 50, 100, 500]:
    print(
        f"Recipes with at least {n} reviews:",
        (reviews_per_recipe >= n).sum()
    )


Recipes with at least 1 reviews: 231637
Recipes with at least 5 reviews: 53593
Recipes with at least 10 reviews: 21399
Recipes with at least 50 reviews: 2164
Recipes with at least 100 reviews: 787
Recipes with at least 500 reviews: 48


The number of recipes decreases sharply as the minimum review threshold increases. While all 231,637 recipes have at least one review, only 21,399 have at least 10 reviews and 787 have at least 100.



## 6. Investigating Zero Ratings

The dataset contains a substantial number of interactions with a rating of 0.

Before calculating average ratings, we need to investigate these records and determine how they should be handled. We will inspect a small sample of associated review texts without modifying the original data.


In [10]:

zero_ratings = interactions[interactions["rating"] == 0]

for review in zero_ratings["review"].dropna().sample(
    n=min(8, zero_ratings["review"].dropna().shape[0]),
    random_state=1
):
    print("-", review[:150])
    print()


- Do these cookies freeze well? Can they be rolled and make with cookie cutters?

- I love this bread but decided to try making a variation using 1 1/2 tsp of cinnamon and about 1/2 c chopped nuts.  I now have both a cinnamon nut and 

- We love this chicken too. I use 1/4 tsp salt, 1/4 tsp garlic powder and 1/2 tsp seasoned salt just to change it up sometimes.  You can also add Italia

- This was the worst chicken I ever had.  It made the whole house stink and it wasn't even ediable.  I threw it out

- I'm so excited I found this recipe! I can't wait to try it out!!!! I was an exchange student in Costa Rica, and I miss the food more than anything els

- This is THE BEST chocolate chip recipe,,, get raves all the time. I use a tablespoon of batter and bake for just about 10 minutes (until they are ligh

- I rate this 100 stars!  These are truly Gourmet.  I added sweetened cranberries, frosted the top and then sprinkled the larger size coconut.  i also d

- This sounds delish. I will de

We randomly selected 8 interactions with a rating of 0 to examine their review texts.

The sample includes questions, positive comments, and negative opinions. This suggests that a rating of 0 does not necessarily indicate a very poor evaluation, but may represent a missing numerical rating.

However, the sample is too small to draw a definitive conclusion. Further investigation is needed before deciding how to handle zero ratings in the ranking analysis.

In [11]:

# Separate interactions with zero ratings from all other interactions
zero_ratings = interactions[interactions["rating"] == 0]
other_ratings = interactions[interactions["rating"] != 0]

# Identify missing or empty review texts
zero_review_missing = (
    zero_ratings["review"].isna()
    | zero_ratings["review"].str.strip().eq("").fillna(False)
)

other_review_missing = (
    other_ratings["review"].isna()
    | other_ratings["review"].str.strip().eq("").fillna(False)
)

print("Interactions with rating 0:", len(zero_ratings))
print("Zero-rated interactions with missing or empty reviews:", zero_review_missing.sum())
print("Percentage:", round(zero_review_missing.mean() * 100, 2), "%")

print("\nInteractions with non-zero ratings:", len(other_ratings))
print("Non-zero interactions with missing or empty reviews:", other_review_missing.sum())
print("Percentage:", round(other_review_missing.mean() * 100, 2), "%")


Interactions with rating 0: 60847
Zero-rated interactions with missing or empty reviews: 0
Percentage: 0.0 %

Interactions with non-zero ratings: 1071520
Non-zero interactions with missing or empty reviews: 172
Percentage: 0.02 %


**Result:** Among the 60,847 interactions with rating 0, none has a missing or empty review text (0%). Among the 1,071,520 interactions with a non-zero rating, only 172 have no text (0.02%).

**Interpretation:** The share of missing text is practically identical in the two groups, so a missing review does not explain the zero ratings. This check cannot tell whether 0 means "no rating given" or "a very low rating". To understand the meaning of 0, the content of the reviews must be examined directly (next step).

In [12]:

# Inspect a random sample of 30 zero-rated interactions with review text
zero_reviews_with_text = zero_ratings[
    zero_ratings["review"].notna()
    & zero_ratings["review"].str.strip().ne("")
]

sample_size = min(30, len(zero_reviews_with_text))

sample_reviews = zero_reviews_with_text.sample(
    n=sample_size,
    random_state=42
)

for review in sample_reviews["review"]:
    print("-", review[:250])
    print()


- I made this for the Summer 2007 Pick A Chef Contest.  I did not give this any stars since I messed it up by not cooking it long enough.  If it is any consolation, my neighbor still ate all of it that was cooked.  I am going to make this again and rep

- I just made this recipe, followed it to a tee, and it is so runny that I'm baking it as brownies.  Will judge the taste after finishing.  How can cookies be made with these ingredients?

- Because of the problems I had with this recipe due to conversions from metric to US and also ingredient equivalents I am not going to rate it at this point.  Problem 1: I think that the puff pastry sheet may be a different size that the poster used -

- Awesome. I added 2 stalks of celery and a red bell pepper

- The last time I made this, my girlfriend advised me not to remove the ham fat after simmering it in the cider. I did it both ways, and it came out great each time.

- We really enjoyed this salad.  I used cranberry mustard, and added dried 

### Conclusion: Interpretation and Treatment of Zero Ratings

The dataset contains 60,847 interactions with a rating of 0, representing approximately 5.37% of all interactions (1,132,367).

Previous analyses of the Food.com dataset interpret a rating of 0 as an interaction in which the user did not provide a numerical rating, rather than as a negative evaluation. This interpretation is described in the university project *Food.com Interaction and Rating Prediction* ([source](https://harshi-saha.github.io/files/food-com-project.pdf)) and is also adopted in the [Recipe-Data-Analytics project](https://github.com/danielpwarren/Recipe-Data-Analytics).

Based on these sources, this project will treat ratings of 0 as missing numerical ratings. They will therefore be excluded when calculating average ratings and counting valid ratings for recipe ranking.

The original dataset will remain unchanged. A separate filtered DataFrame will be created for ranking calculations, retaining only interactions with ratings from 1 to 5. The review text associated with zero-rated interactions will be preserved in the original dataset for potential future analysis.

**Limitation:** This interpretation is based on previous analyses of the dataset, rather than on official Food.com documentation. It is therefore an assumption adopted for this project's analysis.

7. Decision for the Ranking Analysis

Since this project aims to rank recipes based on numerical user ratings, interactions with a rating of 0 will be excluded from rating averages and valid rating counts.

The original dataset will remain unchanged. A separate filtered DataFrame will be created, keeping only interactions with ratings from 1 to 5. The original review text will be preserved for potential future analysis.

To assess the impact of this decision, we may compare the resulting rankings with alternative approaches to handling zero ratings.

In [13]:

# Keep only interactions with valid numerical ratings (1 to 5)
# The original DataFrame remains unchanged
valid_interactions = interactions[
    interactions["rating"].between(1, 5)
].copy()

# Check the number of interactions before and after filtering
print("Original interactions:", len(interactions))
print("Valid interactions:", len(valid_interactions))
print("Excluded interactions:", len(interactions) - len(valid_interactions))

# Count recipes with at least one valid numerical rating
reviews_per_recipe_valid = valid_interactions.groupby("recipe_id").size()

print("Recipes with at least one valid rating:", len(reviews_per_recipe_valid))
print("Recipes without any valid rating:", len(recipes) - len(reviews_per_recipe_valid))


Original interactions: 1132367
Valid interactions: 1071520
Excluded interactions: 60847
Recipes with at least one valid rating: 226590
Recipes without any valid rating: 5047


Result: After excluding the 60,847 zero-rated interactions, 1,071,520 interactions with valid numerical ratings remain. Of the 231,637 recipes, 226,590 have at least one valid rating, while 5,047 recipes (approximately 2.18%) have only zero-rated interactions.

Consequence: These 5,047 recipes cannot be ranked using average numerical ratings because they have no valid ratings. They will therefore be excluded from the ranking calculations. The original interactions DataFrame remains unchanged, and all subsequent rating-based calculations will use valid_interactions.